In [3]:
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv()

required = ["DB_USER", "DB_PASSWORD", "DB_HOST", "DB_PORT", "DB_NAME"]

missing = []
for name in required:
    if os.getenv(name) is None:
        missing.append(name)

if missing:
    raise RuntimeError(f"Missing environment variables: {', '.join(missing)}")

db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

url = URL.create(
    "postgresql+psycopg2",
    username=db_user,
    password=db_password,
    host=db_host,
    port=int(db_port),
    database=db_name,
)
engine = create_engine(url)

with engine.connect() as conn:
    result = conn.execute(text("SELECT 1")).scalar()
    print("connected successfully, SELECT 1 returned", result)

connected successfully, SELECT 1 returned 1


In [4]:
import random
from datetime import date, timedelta

rng = random.Random(42)

rng.choice(["a", "b", "c"])   # picks one item from a list
rng.randint(1, 5)             # picks a whole number from 1 to 5
f"C{7:03d}"                   # builds the text 'C007'
date(2025, 1, 1) + timedelta(days=100)   # a date 100 days after Jan 1

datetime.date(2025, 4, 11)

In [ ]:
import random
from datetime import date, timedelta
import pandas as pd


def generate_csv(path, n=1000, seed=42):
    rng = random.Random(seed)

    products = {"Laptop": 899.99, "Mouse": 19.99, "Keyboard": 500.41, "CPU": 900,
                "Printer": 655.19, "Pen": 16.9, "Book": 20.99}
    statuses = ["pending", "returned", "ordered", "reversed", "shipped"]
    customers = []
    for i in range(1, 51):
        customers.append(f"C{i:03d}")

    rows = []
    start_date = date(2025, 1, 1)
    for i in range(1, n + 1):
        product = rng.choice(list(products))
        order = {
            "order_id": i,
            "customer_id": rng.choice(customers),
            "product": product,
            "quantity": rng.randint(1, 5),
            "unit_price": products[product],
            "order_date": (start_date + timedelta(days=rng.randint(0, 540))).isoformat(),
            "status": rng.choice(statuses),
        }
        rows.append(order)

    
    for order in rng.sample(rows, 12):
        order["unit_price"] = None

    for order in rng.sample(rows, 3):
        order["quantity"] = order["quantity"] * -1

    for order in rng.sample(rows, 200):
        if rng.random() < 0.5:
            order["status"] = order["status"].upper()
        else:
            order["status"] = order["status"].title()

    for order in rng.sample(rows, 25):
        rows.append(order.copy())

    rng.shuffle(rows)

    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)


generate_csv("orders_raw.csv")

In [6]:
df = pd.read_csv("orders_raw.csv")
print(len(df))                             # 1025
print(df["order_id"].duplicated().sum())   # 25
print(df["unit_price"].isna().sum())       # 12
print((df["quantity"] < 0).sum())          # 3
print(df["status"].value_counts())

1025
25
12
3
status
reversed    176
shipped     170
ordered     166
returned    162
pending     144
SHIPPED      29
Shipped      25
REVERSED     24
Pending      24
ORDERED      23
Ordered      17
PENDING      17
Returned     16
RETURNED     16
Reversed     16
Name: count, dtype: int64


In [7]:
def transform(raw):
    df = raw.copy()
    df = df.drop_duplicates(subset="order_id")
    df["status"] = df["status"].str.strip().str.lower()
    df["order_date"] = pd.to_datetime(df["order_date"], errors= "coerce")
    valid = (df["unit_price"].notna()) & (df["quantity"] > 0) & (df["order_date"].notna())
    rejected = df[~valid]
    clean = df[valid].copy()
    clean["total"] = clean["unit_price"] * clean["quantity"]
    return clean, rejected
raw = pd.read_csv("orders_raw.csv")
clean, rejected = transform(raw) 
dupes_dropped = len(raw) - len(clean) - len(rejected) 

In [8]:
print(len(clean), len(rejected))   
print(dupes_dropped)             
print(clean.dtypes)                

985 15
25
order_id                int64
customer_id               str
product                   str
quantity                int64
unit_price            float64
order_date     datetime64[us]
status                    str
total                 float64
dtype: object


In [9]:
print(len(rejected), len(clean))
print(len(rejected) + len(clean))
print(clean[["quantity", "total", "unit_price"]].head())
print(clean.isna().sum())



15 985
1000
   quantity    total  unit_price
0         4    67.60       16.90
1         1   655.19      655.19
2         3  1965.57      655.19
3         4    67.60       16.90
4         2    39.98       19.99
order_id       0
customer_id    0
product        0
quantity       0
unit_price     0
order_date     0
status         0
total          0
dtype: int64


In [10]:
print(len(df), len(clean))   # 1000 985
print("total" in df.columns, "total" in clean.columns)   # False True

1025 985
False True


In [11]:
from sqlalchemy import ForeignKey, Integer, String, Float, Date
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column
from datetime import date

class Base(DeclarativeBase):
    pass

class Customer(Base):
    __tablename__ = "customers"
    customer_id: Mapped[str] = mapped_column(String, primary_key=True)

class Order(Base):
    __tablename__ = "orders"
    order_id: Mapped[int] = mapped_column(Integer, primary_key=True)
    customer_id: Mapped[str] = mapped_column(ForeignKey("customers.customer_id"))
    product: Mapped[str] = mapped_column(String)
    quantity: Mapped[int] = mapped_column(Integer)
    unit_price: Mapped[float] = mapped_column(Float)
    order_date: Mapped[date] = mapped_column(Date)
    status: Mapped[str] = mapped_column(String)
    total: Mapped[float] = mapped_column(Float)

Base.metadata.create_all(engine)



In [12]:
from sqlalchemy import inspect
print(inspect(engine).get_table_names())

['customers', 'orders']


In [13]:
ids = clean["customer_id"].unique()
print(ids[:5])
print(len(ids))

<StringArray>
['C017', 'C006', 'C043', 'C019', 'C005']
Length: 5, dtype: str
50


In [14]:
customers = []
for cid in ids:
    customers.append({"customer_id": cid})
customers
print(len(customers))
print(customers[:3])

50
[{'customer_id': 'C017'}, {'customer_id': 'C006'}, {'customer_id': 'C043'}]


In [15]:
from sqlalchemy import select, func
from sqlalchemy.dialects.postgresql import insert
from sqlalchemy.orm import Session

with Session(engine) as session, session.begin():
    stmt = insert(Customer).on_conflict_do_nothing()
    session.execute(stmt, customers)

In [16]:
with Session(engine) as session:
    print(session.scalar(select(func.count()).select_from(Customer)))


50


In [17]:
orders_df = clean.copy()
orders_df["order_date"] = orders_df["order_date"].dt.date
orders = orders_df.to_dict("records")
print(len(orders))
print(orders[0])


985
{'order_id': 476, 'customer_id': 'C017', 'product': 'Pen', 'quantity': 4, 'unit_price': 16.9, 'order_date': datetime.date(2025, 8, 8), 'status': 'returned', 'total': 67.6}


In [18]:
with Session(engine) as session, session.begin():
    stmt = insert(Order).on_conflict_do_nothing()
    session.execute(stmt, orders)

with Session(engine) as session:
    print(session.scalar(select(func.count()).select_from(Order)))

985


In [19]:
orders[0]["status"] = "shipped"

In [20]:
with Session(engine) as session, session.begin():
    stmt = insert(Order)
    stmt = stmt.on_conflict_do_update(
        index_elements=["order_id"],
        set_={
            "customer_id": stmt.excluded.customer_id,
            "product": stmt.excluded.product,
            "quantity": stmt.excluded.quantity,
            "unit_price": stmt.excluded.unit_price,
            "order_date": stmt.excluded.order_date,       
            "status": stmt.excluded.status,
            "total": stmt.excluded.total,
        },
    )
    session.execute(stmt, orders)

In [21]:
with Session(engine) as session:
    row = session.get(Order, orders[0]["order_id"])
    print(row.status)

shipped


In [22]:
def load(clean, engine):
    Base.metadata.create_all(engine)

    customers = []
    for cid in clean["customer_id"].unique():
        customers.append({"customer_id": cid})

    orders_df = clean.copy()
    orders_df["order_date"] = orders_df["order_date"].dt.date
    orders = orders_df.to_dict("records")

    with Session(engine) as session, session.begin():
        session.execute(insert(Customer).on_conflict_do_nothing(), customers)

        stmt = insert(Order)
        stmt = stmt.on_conflict_do_update(
            index_elements=["order_id"],
            set_={
                    "customer_id": stmt.excluded.customer_id,
                    "product": stmt.excluded.product,
                    "quantity": stmt.excluded.quantity,
                    "unit_price": stmt.excluded.unit_price,
                    "order_date": stmt.excluded.order_date,
                    "status": stmt.excluded.status,
                    "total": stmt.excluded.total,
                },
        )
        session.execute(stmt, orders) 
        return len(orders)

In [23]:
loaded = load(clean, engine)
print(loaded)

with Session(engine) as session:
    print(session.scalar(select(func.count()).select_from(Customer)))
    print(session.scalar(select(func.count()).select_from(Order)))

985
50
985


In [30]:
from sqlalchemy import select, func

stmt = (
    select(Order.status, func.count().label("n")).group_by("status").order_by(func.count().desc())
)

with Session(engine) as session:
    for status, n in session.execute(stmt):
        print(status, n)

shipped 220
reversed 208
ordered 195
returned 185
pending 177


In [25]:
print(stmt)

SELECT orders.status, count(*) AS n 
FROM orders GROUP BY orders.status ORDER BY count(*) DESC


In [32]:
stmt = (
    select(Order.customer_id, func.sum(Order.total).label("spend"))
    .group_by(Order.customer_id)
    .order_by(func.sum(Order.total).desc())
    .limit(5)
)

with Session(engine) as session:
    for customer_id, spend in session.execute(stmt):
        print(customer_id, round(spend, 2))

C025 52930.07
C016 46337.24
C043 43027.1
C020 41622.77
C045 41270.44


In [33]:
print(stmt)

SELECT orders.customer_id, sum(orders.total) AS spend 
FROM orders GROUP BY orders.customer_id ORDER BY sum(orders.total) DESC
 LIMIT :param_1


In [37]:
month = func.to_char(Order.order_date, "YYYY-MM").label("month")

stmt = (
    select(month, func.sum(Order.total).label("revenue"))
    .group_by(month)
    .order_by(month)
)

with Session(engine) as session:
    for m, revenue in session.execute(stmt):
        print(m, round(revenue, 2))

2025-01 60819.02
2025-02 82647.26
2025-03 69716.11
2025-04 84749.61
2025-05 100726.34
2025-06 61872.35
2025-07 70834.35
2025-08 83264.73
2025-09 86526.43
2025-10 68251.12
2025-11 75369.08
2025-12 71983.86
2026-01 76610.57
2026-02 84249.69
2026-03 70082.62
2026-04 80733.13
2026-05 67604.37
2026-06 50074.68


In [ ]:
total_stmt = select(func.sum(Order.total))     

with Session(engine) as session:
    grand_total = session.scalar(total_stmt)

print("TOTAL", round(grand_total, 2))

TOTAL 1346115.32


In [39]:
print(stmt)

SELECT to_char(orders.order_date, :to_char_1) AS month, sum(orders.total) AS revenue 
FROM orders GROUP BY to_char(orders.order_date, :to_char_1) ORDER BY month


In [38]:
print(clean["total"].sum())

1346115.32


In [44]:
def analyze(engine):
    with Session(engine) as session:

        print("Orders per status")
        status_stmt = (
            select(Order.status, func.count().label("n"))
            .group_by(Order.status)
            .order_by(func.count().desc())
        )
        for status, n in session.execute(status_stmt):
            print(f"  {status:<10} {n}")

        print("Top 5 customers by spend")
        top_stmt = (
            select(Order.customer_id, func.sum(Order.total).label("spend"))
            .group_by(Order.customer_id)
            .order_by(func.sum(Order.total).desc())
            .limit(5)
        )
        for customer_id, spend in session.execute(top_stmt):
            print(f"  {customer_id}  {round(spend, 2):,.2f}")

        print("Revenue per month")
        month = func.to_char(Order.order_date, "YYYY-MM").label("month")
        month_stmt = (
            select(month, func.sum(Order.total).label("revenue"))
            .group_by(month)
            .order_by(month)
        )
        for m, revenue in session.execute(month_stmt):
            print(f"  {m}  {round(revenue, 2):>12,.2f}")

        grand_total = session.scalar(select(func.sum(Order.total)))
        print(f"  TOTAL    {round(grand_total, 2):>12,.2f}")


analyze(engine)

Orders per status
  shipped    220
  reversed   208
  ordered    195
  returned   185
  pending    177
Top 5 customers by spend
  C025  52,930.07
  C016  46,337.24
  C043  43,027.10
  C020  41,622.77
  C045  41,270.44
Revenue per month
  2025-01     60,819.02
  2025-02     82,647.26
  2025-03     69,716.11
  2025-04     84,749.61
  2025-05    100,726.34
  2025-06     61,872.35
  2025-07     70,834.35
  2025-08     83,264.73
  2025-09     86,526.43
  2025-10     68,251.12
  2025-11     75,369.08
  2025-12     71,983.86
  2026-01     76,610.57
  2026-02     84,249.69
  2026-03     70,082.62
  2026-04     80,733.13
  2026-05     67,604.37
  2026-06     50,074.68
  TOTAL    1,346,115.32
